# Exploratory Data Analysis: Fraud_Data

This notebook explores the e-commerce fraud dataset (`Fraud_Data.csv`) to understand data quality, feature distributions, class imbalance, and relationships between features and the fraud target (`class`).

**Goals**
- Profile dataset shape, types, missing values, and duplicates
- Visualize univariate distributions for key numerical and categorical features
- Quantify fraud class imbalance
- Examine bivariate relationships between features and fraud
- Summarize actionable insights for preprocessing and modeling

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_fraud_data
from src.preprocessing import (
    class_imbalance_summary,
    duplicate_check,
    missing_values_summary,
    preprocess_fraud_data,
    summary_statistics,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Load Data

We load the raw CSV first for an honest data-quality profile, then apply the shared preprocessing pipeline for analysis and visualization.

In [ ]:
raw_df = load_fraud_data()
df = preprocess_fraud_data(df=raw_df)

print(f"Raw shape: {raw_df.shape}")
print(f"Cleaned shape: {df.shape}")
df.head()

## 2. Dataset Overview

Inspect structure, data types, missing values, and duplicate rows.

In [ ]:
print("Dataset shape:", raw_df.shape)
print("\nColumn dtypes (raw):")
print(raw_df.dtypes)

overview = pd.DataFrame(
    {
        "dtype": raw_df.dtypes.astype(str),
        "non_null": raw_df.notna().sum(),
        "missing": raw_df.isna().sum(),
        "missing_pct": (raw_df.isna().mean() * 100).round(2),
        "unique": raw_df.nunique(dropna=True),
    }
)
overview

In [ ]:
missing_values_summary(raw_df)

In [ ]:
duplicate_summary = duplicate_check(raw_df)
duplicate_summary

In [ ]:
summary_statistics(df)

## 3. Univariate Distributions

Explore the distribution of key numerical and categorical features on the cleaned dataset.

In [ ]:
numerical_features = ["purchase_value", "age", "ip_address"]
categorical_features = ["source", "browser", "sex"]

fig, axes = plt.subplots(1, len(numerical_features), figsize=(15, 4))
for ax, column in zip(axes, numerical_features):
    sns.histplot(df[column], kde=True, ax=ax, bins=40)
    ax.set_title(f"Distribution of {column}")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(categorical_features), figsize=(15, 4))
for ax, column in zip(axes, categorical_features):
    order = df[column].value_counts().head(8).index
    sns.countplot(data=df, x=column, order=order, ax=ax)
    ax.set_title(f"Count by {column}")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
df["hours_to_purchase"] = (
    df["purchase_time"] - df["signup_time"]
).dt.total_seconds() / 3600

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["hours_to_purchase"], kde=True, bins=50, ax=ax)
ax.set_title("Distribution of hours between signup and purchase")
ax.set_xlabel("Hours to purchase")
plt.show()

## 4. Fraud Class Imbalance

Fraud detection datasets are typically highly imbalanced. We quantify the target distribution before modeling.

In [ ]:
imbalance = class_imbalance_summary(df, target_column="class")
imbalance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="class", ax=axes[0])
axes[0].set_title("Fraud class counts")
axes[0].set_xlabel("class (0=legit, 1=fraud)")

sns.barplot(
    data=imbalance,
    x="class",
    y="pct",
    ax=axes[1],
    palette=["#4C72B0", "#DD8452"],
)
axes[1].set_title("Fraud class percentage")
axes[1].set_ylabel("% of transactions")

plt.tight_layout()
plt.show()

print(f"Imbalance ratio (majority/minority): {imbalance.attrs.get('imbalance_ratio')}")

## 5. Bivariate Analysis: Features vs. Fraud Target

Examine how numerical and categorical features relate to the fraud label.

In [ ]:
def fraud_rate_by(column: str) -> pd.DataFrame:
    """Compute fraud rate grouped by a categorical feature."""
    grouped = (
        df.groupby(column, dropna=False, observed=False)["class"]
        .agg(transactions="count", fraud_cases="sum")
        .reset_index()
    )
    grouped["fraud_rate_pct"] = (grouped["fraud_cases"] / grouped["transactions"] * 100).round(3)
    return grouped.sort_values("fraud_rate_pct", ascending=False)


fraud_rate_by("browser").head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="class", y="purchase_value", ax=axes[0])
axes[0].set_title("Purchase value by fraud class")

sns.boxplot(data=df, x="class", y="age", ax=axes[1])
axes[1].set_title("Age by fraud class")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x="class", y="hours_to_purchase", ax=ax)
ax.set_title("Time from signup to purchase by fraud class")
plt.show()

In [ ]:
numeric_cols = ["purchase_value", "age", "hours_to_purchase", "ip_address"]
correlation = df[numeric_cols + ["class"]].corr(numeric_only=True)

plt.figure(figsize=(7, 5))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix (numeric features + fraud class)")
plt.show()

## 6. Fraud Patterns by Browser, Source, Sex, Age, and Purchase Value

Segment-level analysis to identify channels and customer profiles with elevated fraud risk.

In [ ]:
pattern_tables = {
    "browser": fraud_rate_by("browser"),
    "source": fraud_rate_by("source"),
    "sex": fraud_rate_by("sex"),
}

for name, table in pattern_tables.items():
    print(f"\nFraud rate by {name}:")
    display(table)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, column in zip(axes, ["browser", "source", "sex"]):
    rate_df = fraud_rate_by(column)
    sns.barplot(data=rate_df, x=column, y="fraud_rate_pct", ax=ax)
    ax.set_title(f"Fraud rate by {column}")
    ax.set_ylabel("Fraud rate (%)")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
df["age_band"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 45, 55, 100],
    labels=["<=25", "26-35", "36-45", "46-55", "56+"],
    include_lowest=True,
)
df["purchase_value_band"] = pd.qcut(df["purchase_value"], q=4, duplicates="drop")

age_fraud = fraud_rate_by("age_band")
value_fraud = fraud_rate_by("purchase_value_band")

display(age_fraud)
display(value_fraud)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(data=age_fraud, x="age_band", y="fraud_rate_pct", ax=axes[0])
axes[0].set_title("Fraud rate by age band")
axes[0].set_ylabel("Fraud rate (%)")

sns.barplot(data=value_fraud, x="purchase_value_band", y="fraud_rate_pct", ax=axes[1])
axes[1].set_title("Fraud rate by purchase value quartile")
axes[1].set_ylabel("Fraud rate (%)")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# Key metrics used in the written summary below
overall_fraud_rate = df["class"].mean() * 100
imbalance_ratio = imbalance.attrs.get("imbalance_ratio")
median_hours_legit = df.loc[df["class"] == 0, "hours_to_purchase"].median()
median_hours_fraud = df.loc[df["class"] == 1, "hours_to_purchase"].median()
top_source = fraud_rate_by("source").iloc[0]
top_browser = fraud_rate_by("browser").iloc[0]

print(f"Overall fraud rate: {overall_fraud_rate:.2f}%")
print(f"Class imbalance ratio: {imbalance_ratio}:1")
print(f"Median hours to purchase (legit): {median_hours_legit:,.1f}")
print(f"Median hours to purchase (fraud): {median_hours_fraud:.4f}")
print(f"Highest-risk source: {top_source['source']} ({top_source['fraud_rate_pct']:.2f}%)")
print(f"Highest-risk browser: {top_browser['browser']} ({top_browser['fraud_rate_pct']:.2f}%)")

## 7. Key Insights Summary

1. **Severe class imbalance** — Fraud accounts for about **9.4%** of transactions (~14.2k fraud vs ~137k legitimate), with an imbalance ratio near **9.7:1**. Accuracy alone will be misleading; precision-recall, F1, and recall-focused metrics are essential.

2. **Near-instant purchases are a strong fraud signal** — Fraudulent users have a median time from signup to purchase of essentially **0 hours**, while legitimate users wait roughly **1,443 hours (~60 days)**. `hours_to_purchase` (or signup/purchase timestamps) should be a priority feature in modeling.

3. **Acquisition channel matters** — **Direct** traffic shows the highest fraud rate (~**10.5%**), followed by Ads (~9.2%) and SEO (~8.9%). Source-based risk scoring or segmented models may improve detection.

4. **Browser and sex show modest but useful differences** — **Chrome** has a slightly higher fraud rate than Safari and Firefox, and **male** users show a marginally higher rate than female users. These categorical features are worth encoding and testing in combination with temporal features.

5. **Clean raw data, limited univariate separation on age and purchase value** — The dataset has **no missing values** and **no duplicate rows** after inspection. Median **age** (~33) and **purchase value** (~35) are similar across classes, so fraud is better captured through behavioral timing and channel features than single-variable amount/age thresholds.